<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/graphRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install neo4j

In [16]:
from neo4j import GraphDatabase

URI = "neo4j+s://51612bdc.databases.neo4j.io"
USERNAME = "51612bdc"
PASSWORD = "2TciH7DRV-9AAbq_yaPyJBGum1aLMbBgsen9VxKP6lM"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [17]:



def create_nodes(tx):
    tx.run("""
        CREATE (python:Technology {name: "Python"})
        CREATE (ml:Technology {name: "Machine Learning"})
        CREATE (numpy:Library {name: "NumPy"})
        CREATE (pandas:Library {name: "Pandas"})
    """)


def create_relationships(tx):
    tx.run("""
        MATCH (python:Technology {name: "Python"})
        MATCH (ml:Technology {name: "Machine Learning"})
        MATCH (numpy:Library {name: "NumPy"})
        MATCH (pandas:Library {name: "Pandas"})

        CREATE (python)-[:USED_FOR]->(ml)
        CREATE (ml)-[:USES]->(numpy)
        CREATE (ml)-[:USES]->(pandas)
    """)


with driver.session() as session:
    session.execute_write(create_nodes)
    session.execute_write(create_relationships)

print("Graph created!")

driver.close()

Graph created!


In [18]:
from neo4j import GraphDatabase

URI = "neo4j+s://51612bdc.databases.neo4j.io"
USERNAME = "51612bdc"
PASSWORD = "2TciH7DRV-9AAbq_yaPyJBGum1aLMbBgsen9VxKP6lM"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected!")

Connected!


In [22]:
def get_libraries(tx):
    result = tx.run("""
        MATCH (ml:Technology {name: "Machine Learning"})
              -[:USES]->
              (library)
        RETURN library.name AS library
    """)

    return [record["library"] for record in result]


with driver.session() as session:
    libraries = session.execute_read(get_libraries)

print("Libraries used by Machine Learning:")

for library in libraries:
    print(library)

Libraries used by Machine Learning:
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
NumPy
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
Pandas
NumPy
NumPy
NumPy
NumPy
NumP

In [23]:
def get_related_nodes(tx, source_name):
    result = tx.run("""
        MATCH (source {name: $source_name})
              -[r]->
              (target)
        RETURN type(r) AS relationship, target.name AS target
    """, source_name=source_name)

    return [
        {
            "relationship": record["relationship"],
            "target": record["target"]
        }
        for record in result
    ]


source = "Machine Learning"

with driver.session() as session:
    results = session.execute_read(
        get_related_nodes,
        source
    )

print(results)

[{'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'Pandas'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, {'relationship': 'USES', 'target': 'NumPy'}, 

In [24]:
!pip install -q transformers accelerate sentence-transformers neo4j

In [25]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded!")